# Processing Katydid Spectrograms--Multiple Burst Types

Some katydid species alternate between different kinds of bursts in one
recording--a pattern like A, B, C, B, A--that the single-burst pipeline has
no way to see, since it collapses everything into one summary. This is a
standalone variant that reuses the same image generation and detection, then
adds burst-type clustering (group bursts by count/length/spacing) and a
rescue pass for bursts too quiet to clear the global threshold.

Produces one row per (file, burst type)--see the README for the columns.
Stored separately as `katydid_final_multi`; doesn't touch
`Processing_Katydid_Spectrograms.ipynb` or `Display_Function.ipynb`.

In [ ]:
import re
import statistics
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Root directory: one subfolder per species, holding both the audio downloaded by
# Webscraping.ipynb and the oscillograms this notebook generates from it.
KATYDIDS_DIR = Path.home() / 'Discrete_Signals' / 'Katydids'

## Constants

In [ ]:
# Height fraction excluded around the centerline when measuring ink. Thin, so
# quiet near-centerline elements still register; 0.20 is a tuned compromise
CENTER_EXCLUDE_FRACTION = 0.20

SAVE_DPI = 150

# Generated image width = duration * this, so pixel_time stays fine enough to
# resolve fast trills regardless of recording length.
TARGET_PIXELS_PER_SECOND = 600

MIN_FIGURE_WIDTH_INCHES = 4   # floor so very short clips aren't unreadably small

# Silence kept on each side of the first/last pulse when cropping lead-in/out.
CROP_PAD_SECONDS = 0.15


## Helper Functions

In [ ]:
def gaussian_filter1d(signal, sigma):
    """Apply a 1-D Gaussian smoothing kernel to a 1-D array."""
    kernel_radius    = int(4 * sigma + 0.5)
    kernel_positions = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel           = np.exp(-0.5 * (kernel_positions / sigma) ** 2)
    kernel          /= kernel.sum()
    return np.convolve(signal, kernel, mode='same')


def silhouette(values, cluster_labels):
    """Mean silhouette score for a 1-D binary clustering."""
    total_score = 0.0
    for i, value in enumerate(values):
        same_cluster  = values[cluster_labels == cluster_labels[i]]
        other_cluster = values[cluster_labels != cluster_labels[i]]
        within_dist   = np.mean(np.abs(same_cluster  - value)) if len(same_cluster)  > 1 else 0.0
        between_dist  = np.mean(np.abs(other_cluster - value)) if len(other_cluster) > 0 else 0.0
        denom         = max(within_dist, between_dist)
        total_score  += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(values)


def kmeans2(values):
    """Best 2-way split of a 1-D array by exhaustive search over split points.
    Returns (cluster_labels, cluster_centers); label 0 = short, 1 = long."""
    sorted_values    = np.sort(values)
    best_silhouette  = -2.0
    best_split_index = 1

    for i in range(1, len(sorted_values)):
        if i > 1 and sorted_values[i] == sorted_values[i - 1]:
            continue
        split_threshold  = (sorted_values[i - 1] + sorted_values[i]) / 2
        cluster_labels   = (values > split_threshold).astype(int)
        if len(set(cluster_labels)) < 2:
            continue
        split_silhouette = silhouette(values, cluster_labels)
        if split_silhouette > best_silhouette:
            best_silhouette  = split_silhouette
            best_split_index = i

    split_threshold = (sorted_values[best_split_index - 1] + sorted_values[best_split_index]) / 2
    cluster_labels  = (values > split_threshold).astype(int)
    cluster_centers = np.array([
        values[cluster_labels == 0].mean() if (cluster_labels == 0).any() else sorted_values[0],
        values[cluster_labels == 1].mean() if (cluster_labels == 1).any() else sorted_values[-1],
    ])
    return cluster_labels, cluster_centers


def choose_k(gap_durations):
    """1 if the gaps form one cluster, 2 if they split cleanly (silhouette > 0.3)."""
    gap_durations = np.asarray(gap_durations, dtype=float)
    if len(gap_durations) < 3 or len(np.unique(gap_durations)) < 2:
        return 1
    cluster_labels, _ = kmeans2(gap_durations)
    if len(set(cluster_labels)) < 2:
        return 1
    return 2 if silhouette(gap_durations, cluster_labels) > 0.3 else 1


def rle(signal_list):
    """Run-length encode a 1-D sequence."""
    run_list      = []
    current_value = signal_list[0]
    run_length    = 1
    for value in signal_list[1:]:
        if value == current_value:
            run_length += 1
        else:
            run_list.append((current_value, run_length))
            current_value = value
            run_length    = 1
    run_list.append((current_value, run_length))
    return run_list

## Spectrogram Generation

Same idea as the frog pipeline: turn each katydid's downloaded audio into a
clean oscillogram PNG. Crop out lead-in/lead-out silence first (but keep any
inter-burst gaps--those are what we're measuring), find the dominant
frequency, bandpass filter around it, and save a filled waveform with no axes.

Image width scales with the cropped duration, so resolution isn't wasted on
silence and stays fine enough to resolve closely-spaced chirps even in long
recordings.

In [ ]:
def audio_id_from_filename(filename):
    """'Aglaothorax_khioneos_audio_sound.mp3' -> 'sound' (falls back to the stem)."""
    match = re.search(r'_audio_(.+)\.[^.]+$', str(filename))
    return match.group(1) if match else Path(filename).stem


def generated_spectrogram_path(audio_path):
    """Where a katydid's generated oscillogram PNG lives, derived from its audio
    path so the processing function and the batch loop always agree:
    '..._audio_sound.mp3' -> '..._generated_spectrogram_sound.png'."""
    audio_path     = Path(audio_path)
    species_folder = audio_path.parent
    species_name   = species_folder.name
    file_id        = audio_id_from_filename(audio_path.name)
    return species_folder / f'{species_name}_generated_spectrogram_{file_id}.png'


def find_dominant_frequency(signal, sample_rate):
    """Dominant FFT frequency of the clip above 500 Hz (the floor rejects mains
    hum and low-frequency environmental noise)."""
    spectrum  = np.abs(np.fft.rfft(signal))
    freqs     = np.fft.rfftfreq(len(signal), d=1 / sample_rate)
    above_500 = freqs > 500
    peak_idx  = np.argmax(spectrum[above_500])
    return freqs[above_500][peak_idx]


def compute_burst_frequencies(bursts, audio_signal, sample_rate, time_offset=0.0,
                               min_duration_s=0.02, min_range_hz=150.0):
    """Per-burst dominant frequency from the raw audio, as an optional 4th
    clustering feature in classify_burst_types--pitch distinguishes some
    burst types that count/length/spacing can't. Two guards against noise
    faking a distinction: a too-short burst falls back to the whole-clip
    frequency, and if all bursts land within min_range_hz the feature is
    dropped for this file (returns None)."""
    min_samples = max(1, round(min_duration_s * sample_rate))
    whole_clip_freq = find_dominant_frequency(audio_signal, sample_rate)
    freqs = []
    for b in bursts:
        start_sample = max(0, int((time_offset + b['start_time']) * sample_rate))
        end_sample   = min(len(audio_signal), int((time_offset + b['end_time']) * sample_rate))
        segment = audio_signal[start_sample:end_sample]
        freqs.append(find_dominant_frequency(segment, sample_rate) if len(segment) >= min_samples
                     else whole_clip_freq)
    if max(freqs) - min(freqs) < min_range_hz:
        return None
    return freqs


def compute_signal_crop(signal, sample_rate, pad_seconds=CROP_PAD_SECONDS):
    """Sample range from the first to the last audible pulse, padded by
    pad_seconds. Trims only lead-in/lead-out silence--inter-burst gaps between
    the first and last pulse are the signal being measured. The threshold is
    anchored to the noise floor alone (not the eventual peak), so quiet onset
    pulses on crescendo calls survive. Falls back to the
    full signal when there's too little contrast. Returns (start_sample, end_sample)."""
    frame_length = 2048
    hop_length   = 512
    rms = librosa.feature.rms(y=signal, frame_length=frame_length, hop_length=hop_length)[0]

    noise_floor = np.percentile(rms, 10)
    if noise_floor <= 0:
        return 0, len(signal)

    threshold     = noise_floor * 2.0
    active_frames = np.where(rms >= threshold)[0]
    if not len(active_frames):
        return 0, len(signal)

    pad_samples  = int(pad_seconds * sample_rate)
    start_sample = max(0, active_frames[0] * hop_length - pad_samples)
    end_sample   = min(len(signal), active_frames[-1] * hop_length + frame_length + pad_samples)
    return start_sample, end_sample


def generate_spectrogram_image(signal, sample_rate, output_path, band_width_hz=500):
    """Bandpass-filter an already-cropped katydid clip around its dominant
    frequency (+/- band_width_hz; species-tunable via BAND_WIDTH_HZ_OVERRIDES)
    and save a filled, axis-free oscillogram PNG. Figure width scales with
    duration (TARGET_PIXELS_PER_SECOND) so pixel_time stays fine for dense trills."""
    duration_seconds = len(signal) / sample_rate

    # ── Bandpass filter around dominant frequency ──────────────────────────────
    dominant_freq = find_dominant_frequency(signal, sample_rate)

    # Use STFT to isolate the frequency band, then reconstruct via inverse STFT
    stft_matrix = librosa.stft(signal)
    magnitude   = np.abs(stft_matrix)
    phase       = np.angle(stft_matrix)
    frequencies = librosa.fft_frequencies(sr=sample_rate)

    freq_mask          = (
        (frequencies >= dominant_freq - band_width_hz) &
        (frequencies <= dominant_freq + band_width_hz)
    )
    magnitude_filtered = magnitude * freq_mask[:, np.newaxis]

    # Suppress the quietest 80% of the filtered magnitude (remove residual noise)
    noise_floor = np.percentile(magnitude_filtered[magnitude_filtered > 0], 80)
    magnitude_filtered[magnitude_filtered < noise_floor] = 0

    filtered_signal = librosa.istft(magnitude_filtered * np.exp(1j * phase))

    # ── Draw and save oscillogram ──────────────────────────────────────────────
    time_axis = np.linspace(0, len(filtered_signal) / sample_rate, len(filtered_signal))

    figure_width_inches = max(MIN_FIGURE_WIDTH_INCHES, duration_seconds * TARGET_PIXELS_PER_SECOND / SAVE_DPI)

    fig, ax = plt.subplots(figsize=(figure_width_inches, 2))
    ax.plot(time_axis, filtered_signal, color='black', linewidth=0.5)
    ax.fill_between(time_axis, filtered_signal, alpha=1.0, color='black')
    # Pin the x-axis to [0, duration]; otherwise matplotlib's autoscale margin
    # is baked into the PNG by bbox_inches='tight' and every downstream
    # pixel_time-derived duration comes out ~9% short.
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)

    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

## Signal Analysis

A few things tuned specifically for katydids:

- Ink excludes only a thin band around the detected centerline, not a wide
  one--an earlier, wider exclusion band silently dropped real quiet elements.
- Smoothing is narrower than the frog pipeline's (`sigma = 0.15`), since a
  wider kernel started blurring out real short gaps once resolution went up.
- 1-pixel gaps get filled only in dense trills, so real noise blips don't
  fracture one continuous trill into several fake elements.
- The active-fraction ceiling is a bit looser than crickets'--some katydid
  trills genuinely fill more of the recording.
- Thresholding tries a standard ladder first, then an extended one for
  high-baseline recordings, then falls back to treating the whole thing as one
  continuous trill. See `detect_signal_list_adaptive`'s docstring.

In [ ]:
def clean_signal_runs(signal_list, pixel_time, fill_gap_size=0):
    """
    Remove spuriously short on-runs (elements shorter than 3 ms).

    fill_gap_size=0  Never fill off-gaps.
    fill_gap_size>0  Fill off-gaps of <= fill_gap_size pixels surrounded by
                     on-runs on both sides.  Used for dense katydid trills
                     where single-pixel gaps may be image noise.
    """
    signal_list   = np.asarray(signal_list, dtype=np.uint8)
    min_on_pixels = max(1, round(0.003 / pixel_time))

    cleaned = []
    for value, run_length in rle(signal_list):
        if value == 1 and run_length < min_on_pixels:
            cleaned.extend([0] * run_length)
        else:
            cleaned.extend([int(value)] * run_length)
    cleaned = np.array(cleaned, dtype=np.uint8)

    if fill_gap_size > 0 and len(cleaned) > 2:
        result_list = []
        run_list    = rle(cleaned)
        for i, (value, run_length) in enumerate(run_list):
            surrounded_by_signal = (
                i > 0
                and i < len(run_list) - 1
                and run_list[i - 1][0] == 1
                and run_list[i + 1][0] == 1
            )
            if value == 0 and surrounded_by_signal and run_length <= fill_gap_size:
                result_list.extend([1] * run_length)
            else:
                result_list.extend([int(value)] * run_length)
        cleaned = np.array(result_list, dtype=np.uint8)

    return cleaned


def find_centerline_row(spec_array):
    """Row of the always-dark zero-amplitude baseline, found by darkness rather
    than assumed to be the image's middle (a tight bbox can crop asymmetrically)."""
    row_dark_fraction = (spec_array < 200).mean(axis=1)
    return int(np.argmax(row_dark_fraction))


def extract_outer_band_ink(spec_array, center_exclude_fraction=None):
    """2-D ink array with only a thin band around the centerline excluded, so
    quiet near-baseline elements still register. center_exclude_fraction
    overrides the global default (see CENTER_EXCLUDE_FRACTION_OVERRIDES)."""
    if center_exclude_fraction is None:
        center_exclude_fraction = CENTER_EXCLUDE_FRACTION
    img_height, img_width = spec_array.shape
    center_row   = find_centerline_row(spec_array)
    half_exclude = max(1, int(img_height * center_exclude_fraction / 2))
    rows         = np.arange(img_height)
    keep_mask    = np.abs(rows - center_row) > half_exclude
    band         = spec_array[keep_mask, :].astype(float)

    # Ink = darkness relative to background (white background -> near-zero ink in gaps)
    background_level = np.percentile(band, 95)
    ink              = np.clip(background_level - band, 0, None)
    ink_peak         = np.percentile(ink, 99)
    if ink_peak > 0:
        ink /= ink_peak

    return ink


def detect_signal_list_adaptive(ink_array, pixel_time, fill_gap_size_override=None):
    """Threshold the ink array into a binary on/off signal. Walks a standard
    ladder, then an extended high-baseline one, then falls back to a single
    continuous on-run. Returns (signal_list, normalized_signal)."""
    col_pct85           = np.percentile(ink_array, 85, axis=0)
    col_pct95           = np.percentile(ink_array, 95, axis=0)
    col_pct99           = np.percentile(ink_array, 99, axis=0)
    col_signal_strength = gaussian_filter1d(
        0.20 * col_pct85 + 0.35 * col_pct95 + 0.45 * col_pct99,
        sigma=0.15  # narrowed from 0.45 
    )

    signal_floor      = np.percentile(col_signal_strength,  5)
    signal_peak       = np.percentile(col_signal_strength, 99.5)
    if signal_peak <= signal_floor:
        raise ValueError('No contrast in ink array')
    normalized_signal = np.clip(
        (col_signal_strength - signal_floor) / (signal_peak - signal_floor), 0, 1
    )

    min_on_pixels = max(1, round(0.003 / pixel_time))
    # Fill 1-pixel off-gaps only for fine-grained images (dense trills), unless a
    # species-specific override is given (see FILL_GAP_SIZE_OVERRIDES)
    fill_gap_size = fill_gap_size_override if fill_gap_size_override is not None else (
        1 if pixel_time <= 0.004 else 0
    )

    def try_ladder(ladder):
        candidates = []
        for high_threshold, low_threshold in ladder:
            high_mask = normalized_signal >= high_threshold
            low_mask  = normalized_signal >= low_threshold

            signal_list = np.zeros_like(low_mask, dtype=np.uint8)
            column_pos  = 0
            for value, run_length in rle(low_mask.astype(np.uint8)):
                run_start, run_end = column_pos, column_pos + run_length
                if value == 1 and run_length >= min_on_pixels and np.any(high_mask[run_start:run_end]):
                    signal_list[run_start:run_end] = 1
                column_pos = run_end

            signal_list     = clean_signal_runs(signal_list, pixel_time, fill_gap_size)
            active_fraction = float(signal_list.mean())

            if active_fraction <= 0 or active_fraction >= 0.95:
                continue

            on_run_lengths = [run_length for value, run_length in rle(signal_list) if value == 1]
            if not on_run_lengths:
                continue

            median_on_length  = float(np.median(on_run_lengths))
            tiny_run_fraction = sum(l <= 1 for l in on_run_lengths) / len(on_run_lengths)
            score = high_threshold - 0.15 * tiny_run_fraction - 0.002 * median_on_length
            candidates.append((score, signal_list))
        return candidates

    standard_ladder = [(t, max(0.06, t * 0.45)) for t in
                        [0.78, 0.70, 0.62, 0.54, 0.46, 0.38, 0.30, 0.22, 0.16, 0.10]]
    candidate_thresholds = try_ladder(standard_ladder)

    if not candidate_thresholds:
        extended_ladder = [(t, max(0.06, t - 0.12)) for t in [0.98, 0.94, 0.90, 0.86, 0.82]]
        candidate_thresholds = try_ladder(extended_ladder)

    if not candidate_thresholds:
        # Every threshold in both ladders came back too dense (>= 0.95 active):
        # a genuine continuous trill, not a detection failure. Fall back to one
        # long on-run at a low threshold with no active-fraction ceiling.
        low_threshold = 0.10
        low_mask      = normalized_signal >= low_threshold
        signal_list   = clean_signal_runs(low_mask.astype(np.uint8), pixel_time, fill_gap_size)
        if signal_list.mean() <= 0:
            raise ValueError('No valid signal detected at any threshold')
        return signal_list, normalized_signal

    candidate_thresholds.sort(key=lambda t: t[0], reverse=True)
    return candidate_thresholds[0][1], normalized_signal


## Low-Volume Burst Rescue

Ink and signal thresholds are normalized against the whole image, which works
fine until one burst is much quieter than another in the same recording--the
quiet one can get compressed toward zero and never cross the threshold at
all. That matters more here than in the single-burst pipeline: a whole burst
type going missing looks identical to that type never existing.

`rescue_quiet_bursts` catches this by re-checking whatever the global pass
missed, and if there's real (if faint) ink there, re-cropping just that
region and re-running detection locally, judged against its own loudness.

In [ ]:
def group_consecutive(indices, max_gap_px):
    """
    Group a sorted 1-D array of column indices into contiguous regions,
    merging regions separated by at most max_gap_px columns.

    Returns a list of (start, end) tuples, inclusive on both ends.
    """
    if not len(indices):
        return []
    regions = []
    start = prev = indices[0]
    for idx in indices[1:]:
        if idx - prev <= max_gap_px:
            prev = idx
        else:
            regions.append((start, prev))
            start = prev = idx
    regions.append((start, prev))
    return regions


def rescue_quiet_bursts(spec_array, ink, raw_signal_list, pixel_time,
                         center_exclude_fraction=None,
                         presence_threshold=0.03, max_gap_px=8,
                         min_region_px=4, pad_px=3):
    """Recover bursts real but too quiet (vs. a louder burst in the same
    recording) to clear the global threshold; re-crops and re-runs detection on
    each missed region. Returns raw_signal_list OR'd with
    anything rescued."""
    combined = raw_signal_list.copy()
    presence_cols = np.where(ink.max(axis=0) > presence_threshold)[0]
    missed_cols = presence_cols[combined[presence_cols] == 0]
    if not len(missed_cols):
        return combined

    for region_start, region_end in group_consecutive(missed_cols, max_gap_px):
        if region_end - region_start + 1 < min_region_px:
            continue

        pad_start = max(0, region_start - pad_px)
        pad_end   = min(spec_array.shape[1], region_end + pad_px + 1)

        # Already-detected columns touching this region's padded edges mean the
        # global pass DID find something right next to it -- likely this is that
        # same burst's quiet onset/tail, already counted, not a separate burst.
        if combined[pad_start:pad_end].any():
            continue

        crop = spec_array[:, pad_start:pad_end]
        try:
            local_ink = extract_outer_band_ink(crop, center_exclude_fraction=center_exclude_fraction)
            local_signal, _ = detect_signal_list_adaptive(local_ink, pixel_time)
        except ValueError:
            continue  # local region too flat/small to threshold at all -- leave undetected

        combined[pad_start:pad_end] = np.maximum(combined[pad_start:pad_end], local_signal)

    return combined


## Interval Classification & Burst-Type Clustering

Finding burst boundaries works exactly like the single-burst notebook--the
same gap clustering, just refactored (`segment_bursts`) to hand back each
burst's own stats instead of collapsing everything into one summary.

What's new here: a species can alternate between different kinds of bursts
in one recording (say, A, B, C, B, A). `classify_burst_types` clusters those
bursts the same way the gaps were clustered--k-means + silhouette scoring--just generalized to handle more than one feature and more than two groups.

In [ ]:
def silhouette_nd(features, cluster_labels):
    """
    Mean silhouette score for a multi-dimensional clustering -- same idea as
    silhouette() above (cohesion within a cluster vs. separation from other
    clusters), generalized to Euclidean distance so it works on
    multi-feature burst vectors instead of single gap durations.
    """
    total_score = 0.0
    for i, point in enumerate(features):
        same_cluster  = features[cluster_labels == cluster_labels[i]]
        other_cluster = features[cluster_labels != cluster_labels[i]]
        within_dist  = (
            np.mean(np.linalg.norm(same_cluster - point, axis=1))
            if len(same_cluster) > 1 else 0.0
        )
        between_dist = (
            np.mean(np.linalg.norm(other_cluster - point, axis=1))
            if len(other_cluster) > 0 else 0.0
        )
        denom = max(within_dist, between_dist)
        total_score += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(features)


def kmeans_nd(features, k, n_init=8, max_iter=100, random_seed=0):
    """Random-restart Lloyd's k-means for multi-dimensional points (kmeans2's
    exhaustive 1-D search doesn't generalize past one feature / two clusters).
    Returns one cluster label per row of `features`."""
    rng = np.random.default_rng(random_seed)
    n = len(features)
    best_labels, best_inertia = None, np.inf

    for _ in range(n_init):
        centroid_idx = rng.choice(n, size=k, replace=False)
        centroids = features[centroid_idx].copy()

        for _ in range(max_iter):
            distances = np.linalg.norm(features[:, None, :] - centroids[None, :, :], axis=2)
            labels = distances.argmin(axis=1)

            new_centroids = centroids.copy()
            for cluster_id in range(k):
                members = features[labels == cluster_id]
                if len(members):
                    new_centroids[cluster_id] = members.mean(axis=0)

            if np.allclose(new_centroids, centroids):
                centroids = new_centroids
                break
            centroids = new_centroids

        distances = np.linalg.norm(features[:, None, :] - centroids[None, :, :], axis=2)
        labels = distances.argmin(axis=1)
        inertia = float(np.sum((features - centroids[labels]) ** 2))

        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels

    return best_labels


def choose_k_burst_types(features, max_k=4, min_cluster_size=2, n_init=8,
                          min_singleton_silhouette=0.85):
    """Multi-dimensional, multi-k choose_k: try k = 2..max_k, keep the best
    silhouette above 0.3, else one cluster. min_cluster_size rejects singleton
    "types" (noise); a guarded exception lets a confirmed 1-vs-many split
    through (RELAXED_SINGLETON_TYPE_SPECIES). Returns
    one cluster label per row of `features`."""
    n = len(features)
    max_k = min(max_k, n // min_cluster_size)

    best_labels, best_score, best_ok = np.zeros(n, dtype=int), -2.0, False
    if n >= 3 and max_k >= 2:
        for k in range(2, max_k + 1):
            labels = kmeans_nd(features, k, n_init=n_init)
            cluster_sizes = np.bincount(labels, minlength=k)
            if (cluster_sizes < min_cluster_size).any():
                continue
            score = silhouette_nd(features, labels)
            if score > 0.3 and score > best_score:
                best_labels, best_score, best_ok = labels, score, True

    # Singleton-outlier exception for how this bar
    # was calibrated against the full dataset.
    if n >= 4:
        labels2 = kmeans_nd(features, 2, n_init=n_init)
        sizes2 = np.bincount(labels2, minlength=2)
        if sizes2.min() == 1:
            score2 = silhouette_nd(features, labels2)
            if score2 > min_singleton_silhouette and score2 > best_score:
                best_labels, best_score, best_ok = labels2, score2, True

    if not best_ok:
        return np.zeros(n, dtype=int)
    return best_labels


In [ ]:
def segment_bursts(time_bucket_list, leading_silence, trailing_silence,
                    element_length, pixel_time, force_trill=False, min_gap_ratio=5.0,
                    force_element_level_split=False):
    """
    Split time_bucket_list into individual bursts, using the same gap
    classification/burst-boundary guards as
    Processing_Katydid_Spectrograms.ipynb's classify_intervals (duplicated
    here since this notebook stands alone) for the
    full guard rationale, including the 3-cluster fallback and the
    last-resort single-gap acceptance below.

    force_element_level_split=True is the opposite of force_trill: for
    species confirmed to have NO real burst-level grouping at all -- just a
    consistent element/silence pattern -- every detected element becomes its
    own single-element burst, with inter_burst_interval=0.0 (a sentinel
    meaning "no burst-level concept applies here") and each micro-burst's
    inter-element interval set to the recording-wide mean gap. See
    ELEMENT_LEVEL_BURST_SPECIES below.

    Unlike classify_intervals, which collapses straight to a single
    recording-wide median/min/max, this returns each burst's own stats -- the
    input to classify_burst_types' clustering below.

    Returns (inter_element_interval, inter_burst_interval, burst_gap_ratio, bursts),
    where bursts is a list of dicts, one per detected burst (or a single dict
    spanning the whole recording if no burst structure was found / force_trill):
        {'element_count': int, 'mean_element_length': float,
         'mean_inter_element_interval': float, 'start_time': float, 'end_time': float}
    """
    def _element_level_bursts():
        on_durs = [d for s, d in time_bucket_list if s == 'On']
        off_durs = [d for s, d in time_bucket_list if s == 'Off']
        mean_gap = float(np.mean(off_durs)) if off_durs else 0.0
        bursts = []
        elapsed = 0.0
        for state, dur in time_bucket_list:
            if state == 'On':
                bursts.append({
                    'element_count': 1,
                    'mean_element_length': dur,
                    'mean_inter_element_interval': mean_gap,
                    'start_time': elapsed,
                    'end_time': elapsed + dur,
                })
            elapsed += dur
        return mean_gap, 0.0, 0.0, bursts

    if force_element_level_split:
        return _element_level_bursts()

    def _whole_recording_as_one_burst():
        on_durs  = [d for s, d in time_bucket_list if s == 'On']
        off_durs = [d for s, d in time_bucket_list if s == 'Off']
        total_time = leading_silence + sum(d for _, d in time_bucket_list) + trailing_silence
        return [{
            'element_count':               len(on_durs),
            'mean_element_length':         float(np.mean(on_durs)) if on_durs else 0.0,
            'mean_inter_element_interval': float(np.mean(off_durs)) if off_durs else 0.0,
            'start_time':                  0.0,
            'end_time':                    total_time,
        }]

    off_durations = np.array(
        [duration for state, duration in time_bucket_list if state == 'Off'],
        dtype=float,
    )

    if len(off_durations) == 0:
        edge_gaps = [x for x in [leading_silence, trailing_silence] if x > 0]
        return 0.0, float(np.mean(edge_gaps)) if edge_gaps else 0.0, 0.0, _whole_recording_as_one_burst()

    if force_trill:
        return float(np.mean(off_durations)), 0.0, 0.0, _whole_recording_as_one_burst()

    if len(off_durations) < 4 or len(np.unique(np.round(off_durations, 6))) < 2:
        return float(np.mean(off_durations)), 0.0, 0.0, _whole_recording_as_one_burst()

    if choose_k(off_durations) != 2:
        return float(np.mean(off_durations)), 0.0, 0.0, _whole_recording_as_one_burst()

    cluster_labels, cluster_centers = kmeans2(off_durations)
    short_gap_durations = off_durations[cluster_labels == 0]
    long_gap_durations  = off_durations[cluster_labels == 1]
    inter_element_mean  = float(np.mean(short_gap_durations))
    inter_burst_mean    = float(np.mean(long_gap_durations))

    if inter_element_mean <= 0:
        return float(np.mean(off_durations)), 0.0, 0.0, _whole_recording_as_one_burst()

    burst_gap_ratio     = inter_burst_mean / inter_element_mean
    absolute_separation = inter_burst_mean - inter_element_mean
    long_gap_fraction   = len(long_gap_durations) / len(off_durations)
    n_long_gaps         = len(long_gap_durations)
    inter_burst_pixels  = inter_burst_mean / pixel_time

    if n_long_gaps == 1:
        single_long_gap_index = int(np.where(cluster_labels == 1)[0][0])
        single_gap_position   = single_long_gap_index / max(1, len(off_durations) - 1)
    else:
        single_gap_position = 0.5

    is_burst = (
        burst_gap_ratio     >= min_gap_ratio
        and inter_burst_mean    >= 2.5 * element_length
        and absolute_separation >= 1.5 * element_length
        and long_gap_fraction   <= 0.55
        and inter_burst_pixels  >= 10
        and (
            n_long_gaps >= 2
            or (inter_burst_pixels >= 200 and single_gap_position >= 0.25)
        )
    )

    if not is_burst and n_long_gaps == 1 and len(short_gap_durations) >= 6:
        if choose_k(short_gap_durations) == 2:
            secondary_labels, _ = kmeans2(short_gap_durations)
            secondary_silhouette = silhouette(short_gap_durations, secondary_labels)
            secondary_short = short_gap_durations[secondary_labels == 0]
            secondary_long  = short_gap_durations[secondary_labels == 1]
            if (secondary_silhouette > 0.5 and len(secondary_long) >= 2
                    and float(np.mean(secondary_long)) >= 4.0 * float(np.mean(secondary_short))
                    and float(np.mean(secondary_long)) / pixel_time >= 5):
                inter_element_mean = float(np.mean(secondary_short))
                inter_burst_mean   = float(np.mean(secondary_long))
                burst_gap_ratio    = inter_burst_mean / inter_element_mean
                long_gap_fraction  = len(secondary_long) / len(short_gap_durations)
                inter_burst_pixels = inter_burst_mean / pixel_time
                n_long_gaps        = len(secondary_long)
                is_burst = (
                    burst_gap_ratio    >= min_gap_ratio
                    and inter_burst_mean   >= 2.5 * element_length
                    and long_gap_fraction  <= 0.55
                    and inter_burst_pixels >= 10
                    and n_long_gaps        >= 2
                )

    # Last-resort single-gap acceptance: drops the position>=0.25 requirement
    # once the gap is unambiguously wide (inter_burst_pixels>=200) -- audited
    # against the full dataset with zero regressions.
    # (Orchelimum_campestre).
    if not is_burst and n_long_gaps == 1 and inter_burst_pixels >= 200:
        is_burst = (
            burst_gap_ratio     >= min_gap_ratio
            and inter_burst_mean    >= 2.5 * element_length
            and absolute_separation >= 1.5 * element_length
            and long_gap_fraction   <= 0.55
        )

    if not is_burst:
        return float(np.mean(off_durations)), 0.0, burst_gap_ratio, _whole_recording_as_one_burst()

    burst_boundary_time = (inter_element_mean + inter_burst_mean) / 2

    bursts               = []
    current_on_durs      = []
    current_internal_gaps = []
    current_start_time   = 0.0
    elapsed              = 0.0

    for state, duration in time_bucket_list:
        if state == 'On':
            if not current_on_durs:
                current_start_time = elapsed
            current_on_durs.append(duration)
        else:  # "Off"
            if duration > burst_boundary_time:
                if current_on_durs:
                    bursts.append({
                        'element_count':               len(current_on_durs),
                        'mean_element_length':          float(np.mean(current_on_durs)),
                        'mean_inter_element_interval':  (
                            float(np.mean(current_internal_gaps)) if current_internal_gaps else 0.0
                        ),
                        'start_time': current_start_time,
                        'end_time':   elapsed,
                    })
                current_on_durs, current_internal_gaps = [], []
            elif current_on_durs:
                # a within-burst gap -- record it, don't end the burst
                current_internal_gaps.append(duration)
        elapsed += duration

    if current_on_durs:
        bursts.append({
            'element_count':               len(current_on_durs),
            'mean_element_length':          float(np.mean(current_on_durs)),
            'mean_inter_element_interval':  (
                float(np.mean(current_internal_gaps)) if current_internal_gaps else 0.0
            ),
            'start_time': current_start_time,
            'end_time':   elapsed,
        })

    if not bursts:
        return float(np.mean(off_durations)), 0.0, burst_gap_ratio, _whole_recording_as_one_burst()

    return inter_element_mean, inter_burst_mean, burst_gap_ratio, bursts


def classify_intervals(time_bucket_list, leading_silence, trailing_silence,
                       element_length, pixel_time, force_trill=False):
    """segment_bursts() collapsed to the single-burst notebook's
    classify_intervals() return shape. Kept for parity; katydid_process_multi
    uses segment_bursts + classify_burst_types directly. Returns
    (inter_element_interval, inter_burst_interval, elements_per_burst,
    min_elements_per_burst, max_elements_per_burst, burst_gap_ratio)."""
    inter_element_interval, inter_burst_interval, burst_gap_ratio, bursts = segment_bursts(
        time_bucket_list, leading_silence, trailing_silence, element_length, pixel_time,
        force_trill=force_trill,
    )
    counts = [b['element_count'] for b in bursts]
    median_epb = int(statistics.median(counts)) if counts else 1
    return (
        inter_element_interval, inter_burst_interval, max(1, median_epb),
        min(counts) if counts else 1, max(counts) if counts else 1, burst_gap_ratio,
    )


### Burst-Type Clustering

`segment_bursts` above only finds where each burst starts and ends--this is
the part that decides whether they're all the same kind of burst or several
alternating kinds.

In [ ]:
def classify_burst_types(bursts, max_k=4, force_uniform=False, min_singleton_silhouette=0.85,
                          burst_frequencies=None):
    """
    Cluster this recording's bursts into distinct "types" (e.g. burst A/B/C in
    an alternating call structure like ABCBA), using the same k-means +
    silhouette technique as the gap classification above, generalized to
    arbitrary-dimensional burst features and more than two clusters
    (kmeans_nd/silhouette_nd/choose_k_burst_types).

    Features per burst: element_count, mean_element_length,
    mean_inter_element_interval, and (if burst_frequencies is given) each
    burst's own dominant frequency in Hz -- z-score normalized so element
    counts, second-scale durations, and Hz-scale frequencies all contribute
    comparably to the distance calculation.

    Mutates and returns `bursts`, with each dict annotated with:
      'burst_type'    int, assigned in order of FIRST APPEARANCE in the
                      recording (not raw cluster index), e.g. [0, 1, 2, 1, 0]
      'burst_pattern' str, the same sequence spelled out with letters
                      (0->A, 1->B, ...), e.g. "ABCBA" -- same across every
                      burst dict in the list

    If there's only one burst, or clustering doesn't find well-separated
    groups, every burst gets burst_type=0 ("A").

    force_uniform=True skips clustering entirely and returns a single uniform
    type -- for UNIFORM_BURST_TYPE_SPECIES, species confirmed by manual
    review to have only one real burst type where ordinary burst-to-burst
    jitter was otherwise read as a type split.
    """
    n = len(bursts)
    if n < 2 or force_uniform:
        for b in bursts:
            b['burst_type']    = 0
            b['burst_pattern'] = 'A' * n
        return bursts

    feature_rows = [
        [b['element_count'], b['mean_element_length'], b['mean_inter_element_interval']]
        for b in bursts
    ]
    if burst_frequencies is not None:
        for row, freq in zip(feature_rows, burst_frequencies):
            row.append(freq)
    features = np.array(feature_rows, dtype=float)

    feature_std = features.std(axis=0)
    feature_std[feature_std == 0] = 1.0  # guard a constant column (e.g. every burst has 1 internal gap)
    features_norm = (features - features.mean(axis=0)) / feature_std

    labels = choose_k_burst_types(
        features_norm, max_k=max_k, min_singleton_silhouette=min_singleton_silhouette,
    )

    first_seen = {}
    relabeled  = []
    for lbl in labels:
        if lbl not in first_seen:
            first_seen[lbl] = len(first_seen)
        relabeled.append(first_seen[lbl])

    pattern = ''.join(chr(ord('A') + t) for t in relabeled)
    for b, t in zip(bursts, relabeled):
        b['burst_type']    = t
        b['burst_pattern'] = pattern

    return bursts


## Audio Cropping for Long / Ambiguous Recordings

A few species have recordings so long, or with so many different scales of
gap (element, burst, verse), that the usual two-cluster gap detection can't
tell inter-element from inter-burst gaps anymore--it'll happily report an
inter-element gap of nearly a second, which is nonsense.

For a known list of species (plus anything that produces a similarly
degenerate result), we crop the audio down to 2-3 clean bursts before ever
generating the oscillogram--same idea as `Frog_Clip_Log.ipynb`, done inline
here instead of as a separate notebook. If no clean burst structure turns up,
it falls back to just grabbing the first 15 elements.

Cropped clips go to `Cropped_Katydids_Audios/`, logged the same way
`frog_clip_log.csv` is, and get substituted in wherever `katydid_process`
would otherwise load the original file.

In [ ]:
# Root folder for manually-cropped audio clips + crop log (mirrors Cropped_Frogs_Audios/).
CROPPED_KATYDIDS_DIR = Path.home() / 'Discrete_Signals' / 'Cropped_Katydids_Audios'
CROPPED_KATYDIDS_DIR.mkdir(exist_ok=True)

# Species whose downloaded recordings are long/multi-scale enough that bimodal
# burst detection breaks down (see markdown above). Flagged by manual inspection.
LONG_RECORDING_SPECIES = [
    'Amblycorypha_longinicta', 'Amblycorypha_rivograndis', 'Insara_covilleae',
    'Amblycorypha_rotundifolia', 'Atlanticus_americanus', 'Atlanticus_calcaratus',
    'Atlanticus_dorsalis', 'Atlanticus_gibbosus', 'Atlanticus_monticola',
    'Bucrates_malivolans', 'Capnobotes_bruneri', 'Capnobotes_fuliginosus',
    'Conocephalus_brevipennis', 'Conocephalus_fasciatus', 'Eremopedes_balli',
    'Neduba_arborea', 'Neduba_cascadia', 'Neduba_inversa', 'Neduba_longiplutea',
    'Neduba_macneilli', 'Neduba_oblongata', 'Neduba_propsti', 'Neduba_radocantans',
    'Scudderia_furcata',
]

# Species confirmed by visual inspection to be one continuous trill, where the
# 3-cluster fallback would otherwise carve out a fake "burst" from ordinary
# amplitude-modulation dips. Forces elements_per_burst=1, inter_burst_interval=0.0.
CONTINUOUS_TRILL_SPECIES = [
    'Idiostatus_hermannii',
    'Amblycorypha_cajuni',
    'Neoconocephalus_melanorhinus',
    'Neduba_sierranus',
    'Neduba_oblongata',
    'Orchelimum_gladiator',
    'Neduba_ambagiosa',
    'Neduba_prorocantans',
    'Conocephalus_brevipennis',
]

# Species-specific fill_gap_size override for detect_signal_list_adaptive
# (the default 1px fill isn't tolerant enough for these ).
FILL_GAP_SIZE_OVERRIDES = {
    'Aglaothorax_tinkhamorum': 4,
    'Neduba_lucubrata': 5,
}

# Species-specific CENTER_EXCLUDE_FRACTION override -- the global 0.20 still
# drops real quiet elements for these species.
CENTER_EXCLUDE_FRACTION_OVERRIDES = {
    'Scudderia_cuneata': 0.05,
    'Inscudderia_strigata': 0.03,
    'Inscudderia_taxodii': 0.03,
    'Idionotus_tehachapi': 0.10,
    'Eremopedes_bilineatus': 0.16,
    'Arethaea_phalangium': 0.10,
}

# Species whose only available SINA recording is explicitly NOT a calling song
# (per katydid_df's Description text), so its parameters aren't comparable to
# every other species' calling-song measurements. Skipped entirely.
EXCLUDED_NON_CALLING_SONG_SPECIES = [
    'Paracyrtophyllus_excelsus',  # katydid_df Description: "14 s of protest song"
]

# Per-species fill_gap_size passed into detect_signal_list_adaptive's ladder
# (distinct from FILL_GAP_SIZE_OVERRIDES, which is post-trim). Eremopedes_covilleae
# has real 1-pixel gaps the default was merging.
LADDER_FILL_GAP_OVERRIDES = {
    'Eremopedes_covilleae': 0,
}

# Species-specific override for segment_bursts' burst_gap_ratio guard
# (default 5.0). (Aglaothorax_hulodomus).
RELAXED_BURST_GAP_RATIO_SPECIES = {
    'Aglaothorax_hulodomus': 3.0,
}

# Species confirmed by manual review to have only one real burst type, where
# classify_burst_types' primary clustering path was reading ordinary
# burst-to-burst jitter as a real type split. Passed to classify_burst_types
# as force_uniform.
UNIFORM_BURST_TYPE_SPECIES = {
    'Aglaothorax_amathitis',
    'Aglaothorax_armiger',
    'Aglaothorax_segnis',
    'Aglaothorax_strobilion',
    'Aglaothorax_tinkhamorum',
    'Amblycorypha_carinata',
    'Atlanticus_dorsalis',
    'Belocephalus_davisi',
    'Capnobotes_occidentalis',
    'Conocephalus_nemoralis',
    'Orchelimum_volantum',
    'Neduba_diabolica',
    'Neduba_lucubrata',
    'Idionotus_brunneus',
    'Eremopedes_covilleae',
    'Belocephalus_sabalis',
    'Obolopteryx_brevihastata',
    'Clinopleura_infuscata',
}

# Species confirmed to have NO real burst-level grouping at all -- just a
# consistent element/silence pattern, not a series of multi-element verses.
# Passed to segment_bursts as force_element_level_split (one burst per element).
ELEMENT_LEVEL_BURST_SPECIES = {
    'Aglaothorax_acrolophitus',
    'Aglaothorax_bufonoides',
    'Aglaothorax_dactyla',
    'Aglaothorax_kelainops',
    'Aglaothorax_poecilonotum',
    'Neduba_sierranus',
    'Neduba_radicata',
    'Insara_covilleae',
    'Amblycorypha_longinicta',
    'Idiostatus_aequalis',
    'Neduba_oblongata',
    'Neduba_sequoia',
    'Neoconocephalus_ensiger',
    'Neduba_duplocantans',
    'Idiostatus_inermis',
    'Conocephalus_attenuatus',
    'Neduba_ambagiosa',
    'Conocephalus_allardi',
    'Conocephalus_aigialus',
    'Anabrus_cerciata',
    'Anabrus_simplex',
    'Belocephalus_sleighti',
}

# Species confirmed to over-split into too many burst types -- real type
# variation exists, just capped at a lower k than the default search finds.
MAX_BURST_TYPES_OVERRIDES = {
    'Orchelimum_delicatum': 2,
}

# Species that need a real singleton/small-group burst-type split that
# choose_k_burst_types' default 0.85 singleton-silhouette bar rejects. See
# that function's docstring and.
RELAXED_SINGLETON_TYPE_SPECIES = {
    'Conocephalus_cinereus': 0.55,
}

# Species-specific band_width_hz override for generate_spectrogram_image
# (default +/-500Hz around the dominant frequency cuts into real signal for
# these species).
BAND_WIDTH_HZ_OVERRIDES = {
    'Amblycorypha_huasteca': 2000,
    'Amblycorypha_parvipennis': 3000,
}


def detect_audio_elements(signal, sample_rate, frame_length=512, hop_length=128, min_element_s=0.003):
    """Coarse RMS-level element detector, used only to pick a crop window (finer
    hop than compute_signal_crop so ~10 ms elements resolve). Not the detector
    behind the reported metrics."""
    rms = librosa.feature.rms(y=signal, frame_length=frame_length, hop_length=hop_length)[0]
    noise_floor = np.percentile(rms, 10)
    threshold = np.percentile(rms, 50) * 0.5 if noise_floor <= 0 else noise_floor * 2.0
    active = rms >= threshold

    raw_runs = []
    in_run, run_start = False, 0
    for i, v in enumerate(active):
        if v and not in_run:
            in_run, run_start = True, i
        elif not v and in_run:
            in_run = False
            raw_runs.append((run_start, i))
    if in_run:
        raw_runs.append((run_start, len(active)))

    min_frames = max(1, int(min_element_s * sample_rate / hop_length))
    raw = [
        (s * hop_length, min(len(signal), e * hop_length + frame_length))
        for s, e in raw_runs if e - s >= min_frames
    ]

    # Merge runs separated by a very brief dip (mid-pulse amplitude modulation,
    # not a real inter-element silence) -- otherwise one physical pulse gets
    # fragmented into several spurious "elements".
    merge_gap_samples = int(0.006 * sample_rate)
    merged = []
    for s, e in raw:
        if merged and s - merged[-1][1] <= merge_gap_samples:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append((s, e))
    return merged


def select_crop_window(elements, sample_rate, total_samples, pad_seconds=0.2,
                        min_elements_fallback=15, max_bursts=3,
                        max_burst_elements=40, max_burst_duration_s=12.0,
                        min_burst_elements=3, dense_passage_min_duration_s=0.3):
    """Pick a (start_sample, end_sample, note) window spanning 2-3 bursts, or the
    first min_elements_fallback elements if there's no clean burst structure
    (same kmeans2/choose_k gap split as classify_intervals, plus
    plausible-burst-size guards). Returns None when already short enough.
    Special case: if the coarse detector merged a dense trill into one giant
    element (> dense_passage_min_duration_s), crop straight to it."""
    n = len(elements)
    if n == 0:
        return None

    dense_passage_idx = next(
        (i for i, (s, e) in enumerate(elements) if (e - s) / sample_rate >= dense_passage_min_duration_s),
        None,
    )
    if n < min_elements_fallback and dense_passage_idx is not None:
        pad = int(pad_seconds * sample_rate)
        s, e = elements[dense_passage_idx]
        start = max(0, s - pad)
        end   = min(total_samples, e + pad)
        duration_s = (e - s) / sample_rate
        return start, end, f'dense passage window (merged element, {duration_s:.2f}s)'

    gaps = np.array([
        (elements[i + 1][0] - elements[i][1]) / sample_rate
        for i in range(n - 1)
    ])

    if n < min_elements_fallback:
        return None  # already concise -- no crop needed

    burst_threshold = None
    if len(gaps) >= 4 and len(np.unique(np.round(gaps, 6))) >= 2 and choose_k(gaps) == 2:
        labels, _ = kmeans2(gaps)
        short_gaps = gaps[labels == 0]
        long_gaps  = gaps[labels == 1]
        if len(long_gaps) >= 4 and long_gaps.max() / max(long_gaps.min(), 1e-9) > 3 and choose_k(long_gaps) == 2:
            labels2, _ = kmeans2(long_gaps)
            burst_threshold = long_gaps[labels2 == 0].max()
        else:
            burst_threshold = short_gaps.max()

    bursts = [[0]]
    if burst_threshold is not None:
        for i, gap in enumerate(gaps):
            if gap > burst_threshold:
                bursts.append([])
            bursts[-1].append(i + 1)
    n_bursts = len(bursts) if burst_threshold is not None else 0

    if n_bursts >= 2:
        burst_sizes = [len(b) for b in bursts]
        burst_durations = [
            (elements[b[-1]][1] - elements[b[0]][0]) / sample_rate for b in bursts
        ]
        if (max(burst_sizes) > max_burst_elements
                or max(burst_durations) > max_burst_duration_s
                or np.median(burst_sizes) < min_burst_elements):
            n_bursts = 0  # not real burst structure -- use element-count fallback

    pad = int(pad_seconds * sample_rate)
    if n_bursts >= 2:
        chosen = bursts[:max_bursts]
        start = max(0, elements[chosen[0][0]][0] - pad)
        end   = min(total_samples, elements[chosen[-1][-1]][1] + pad)
        return start, end, f'burst window (bursts={len(chosen)} of {n_bursts})'
    else:
        take = min(n, min_elements_fallback) if n >= min_elements_fallback else n
        start = max(0, elements[0][0] - pad)
        end   = min(total_samples, elements[take - 1][1] + pad)
        return start, end, f'element window (elements={take}, no clean burst structure)'


In [ ]:
import soundfile as sf

crop_log_path = CROPPED_KATYDIDS_DIR / 'katydid_crop_log.csv'

crop_rows = []
for species_folder in LONG_RECORDING_SPECIES:
    genus, species = species_folder.split('_', 1)
    sp_dir = KATYDIDS_DIR / species_folder
    audio_files = sorted(
        p for p in sp_dir.glob(f'{species_folder}_audio_*')
        if p.suffix.lower() in {'.mp3', '.wav', '.ogg'}
    )
    for audio_num, audio_path in enumerate(audio_files, start=1):
        raw_signal, sample_rate = librosa.load(str(audio_path), sr=None)
        source_dur_s = len(raw_signal) / sample_rate
        elements = detect_audio_elements(raw_signal, sample_rate)
        result = select_crop_window(elements, sample_rate, len(raw_signal))

        if result is None:
            crop_rows.append(dict(
                genus=genus, species=species, audio_num=audio_num,
                source_file=str(audio_path), clipped_file='',
                status='skipped', note=f'no crop needed ({len(elements)} elements)',
                source_dur_s=round(source_dur_s, 3),
                start_time_s=None, end_time_s=None, clip_duration_s=None,
            ))
            continue

        start, end, note = result
        clip = raw_signal[start:end]
        clipped_file = CROPPED_KATYDIDS_DIR / f'{species_folder}_{audio_num}.wav'
        sf.write(clipped_file, clip, sample_rate)
        crop_rows.append(dict(
            genus=genus, species=species, audio_num=audio_num,
            source_file=str(audio_path), clipped_file=str(clipped_file),
            status='ok', note=note,
            source_dur_s=round(source_dur_s, 3),
            start_time_s=round(start / sample_rate, 3),
            end_time_s=round(end / sample_rate, 3),
            clip_duration_s=round(len(clip) / sample_rate, 3),
        ))

katydid_crop_log = pd.DataFrame(crop_rows, columns=[
    'genus', 'species', 'audio_num', 'source_file', 'clipped_file',
    'status', 'note', 'source_dur_s', 'start_time_s', 'end_time_s', 'clip_duration_s',
])
katydid_crop_log.to_csv(crop_log_path, index=False)

# Lookup used by katydid_process: original audio path (str) -> cropped audio path (str).
# Only present for rows that were actually cropped ('ok'); everything else keeps
# using its original, uncropped audio file.
KATYDID_CROP_LOOKUP = {
    row['source_file']: row['clipped_file']
    for row in crop_rows if row['status'] == 'ok'
}

print(f'{len(katydid_crop_log)} files evaluated | '
      f"{(katydid_crop_log.status == 'ok').sum()} cropped | "
      f"{(katydid_crop_log.status == 'skipped').sum()} left as-is")
katydid_crop_log

In [ ]:
# Store for reference elsewhere (mirrors frog pipeline's clips_ready / %store pattern).
%store katydid_crop_log

## Missed-Element Rescue (species-specific)

Image generation itself can erase a real secondary burst before the
oscillogram is even drawn--it picks one dominant frequency and noise floor
for the whole recording, so a quieter or differently-pitched region can get
filtered out entirely. No amount of threshold tuning downstream can recover
signal that was never rendered.

The fix pulls a candidate region straight from the raw audio, runs it through
the normal detection pipeline as its own small crop, and merges the result in
afterward. This only runs for an 11-file whitelist
(`MISSED_ELEMENT_RESCUE_FILES`), each confirmed by hand--telling a genuine
missed element apart from look-alikes (attack ramps, crop-padding residue)
needed a human look each time.

In [ ]:
# (species_folder, audio_filename) pairs confirmed, by manual visual +
# amplitude-ratio verification, to have a real secondary element/burst
# entirely missing from the primary detection. for how
# this list was arrived at -- an automated scan alone isn't trusted here.
MISSED_ELEMENT_RESCUE_FILES = {
    ('Amblycorypha_arenicola', 'Amblycorypha_arenicola_audio_002so.mp3'),
    ('Orchelimum_superbum', 'Orchelimum_superbum_audio_260ss2.mp3'),
    ('Turpilia_rostrata', 'Turpilia_rostrata_audio_071ssa2.mp3'),
    ('Aglaothorax_dactyla', 'Aglaothorax_dactyla_audio_sound.mp3'),
    ('Orchelimum_campestre', 'Orchelimum_campestre_audio_266ss2.mp3'),
    ('Amblycorypha_cajuni', 'Amblycorypha_cajuni_audio_012so.mp3'),
    ('Amblycorypha_insolita', 'Amblycorypha_insolita_audio_016ss.mp3'),
    ('Amblycorypha_rivograndis', 'Amblycorypha_rivograndis_audio_010so.mp3'),
    ('Orchelimum_bullatum', 'Orchelimum_bullatum_audio_264ss2.mp3'),
    ('Orchelimum_gladiator', 'Orchelimum_gladiator_audio_263ss2.mp3'),
    ('Conocephalus_cinereus', 'Conocephalus_cinereus_audio_232ss2.mp3'),
}


def render_raw_crop_png(signal_slice, sample_rate, output_path):
    """Render a rescue-candidate crop straight from the raw signal (not
    generate_spectrogram_image--its STFT gates are what's being bypassed).
    Skips MIN_FIGURE_WIDTH_INCHES so a ~40 ms crop isn't over-sampled into
    losing its short pulses."""
    duration_seconds     = len(signal_slice) / sample_rate
    figure_width_inches  = max(0.3, duration_seconds * TARGET_PIXELS_PER_SECOND / SAVE_DPI)
    time_axis            = np.linspace(0, duration_seconds, len(signal_slice))

    fig, ax = plt.subplots(figsize=(figure_width_inches, 2))
    ax.plot(time_axis, signal_slice, color='black', linewidth=0.5)
    ax.fill_between(time_axis, signal_slice, alpha=1.0, color='black')
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)
    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)


def find_missed_element_regions(signal, sample_rate, primary_trim_sample_range,
                                 presence_multiplier=6.0, win_s=0.002,
                                 cluster_gap_s=0.015, isolation_gap_s=0.08,
                                 edge_buffer_s=0.25, pad_s=0.02,
                                 max_group_span_s=0.2,
                                 min_fraction_of_primary_peak=0.08,
                                 clip_edge_exclude_s=0.17):
    """Regions of raw-audio energy well above ambient, sitting entirely outside
    the primary detected region and separated from it by real quiet--candidate
    missed secondary bursts. Peak (not RMS) amplitude, since elements can be
    ~2 ms. Three guards (max span, min fraction of primary peak, clip-edge
    exclusion) each reject a specific confirmed false positive."""
    win   = max(1, int(win_s * sample_rate))
    n_win = len(signal) // win
    if n_win < 4:
        return []

    raw_env = np.array([
        np.max(np.abs(signal[j * win:(j + 1) * win])) for j in range(n_win)
    ])
    if raw_env.max() <= 0:
        return []

    baseline = np.percentile(raw_env, 10)
    if baseline <= 0:
        baseline = raw_env[raw_env > 0].min() if (raw_env > 0).any() else 1e-6

    ts0, ts1     = primary_trim_sample_range
    edge_buffer  = int(edge_buffer_s * sample_rate)
    win0, win1   = max(0, (ts0 - edge_buffer) // win), min(n_win, (ts1 + edge_buffer) // win + 1)
    outside      = np.ones(n_win, dtype=bool)
    outside[win0:win1] = False

    clip_edge_win = max(1, int(clip_edge_exclude_s / win_s))
    outside[:clip_edge_win] = False
    outside[max(0, n_win - clip_edge_win):] = False

    primary_peak     = np.max(np.abs(signal[ts0:ts1])) if ts1 > ts0 else raw_env.max()
    min_absolute_peak = primary_peak * min_fraction_of_primary_peak

    real_energy = (raw_env > baseline * presence_multiplier) & (raw_env > min_absolute_peak)
    candidate   = real_energy & outside
    if candidate.sum() < 1:
        return []

    gap_win = max(1, int(cluster_gap_s / win_s))
    idxs    = np.where(candidate)[0]
    groups  = []
    cur     = [idxs[0]]
    for i in idxs[1:]:
        if i - cur[-1] <= gap_win:
            cur.append(i)
        else:
            groups.append(cur)
            cur = [i]
    groups.append(cur)

    confirmed          = []
    quiet_thresh        = baseline * 2.0
    min_quiet_run        = max(1, int(isolation_gap_s / win_s))
    max_group_span_win  = max(1, int(max_group_span_s / win_s))
    for g in groups:
        g0, g1 = g[0], g[-1]
        if (g1 - g0 + 1) > max_group_span_win:
            continue
        ok = True
        if g0 >= win1:
            between = raw_env[win1:g0] if g0 > win1 else np.array([])
        elif g1 < win0:
            between = raw_env[g1 + 1:win0] if win0 > g1 else np.array([])
        else:
            between = np.array([])
        if len(between):
            max_quiet_run = 0
            run           = 0
            for v in between:
                if v < quiet_thresh:
                    run           += 1
                    max_quiet_run  = max(max_quiet_run, run)
                else:
                    run = 0
            if max_quiet_run < min_quiet_run:
                ok = False
        if ok:
            confirmed.append((g0, g1))

    pad     = int(pad_s * sample_rate)
    regions = []
    for g0, g1 in confirmed:
        s0 = max(0, g0 * win - pad)
        s1 = min(len(signal), (g1 + 1) * win + pad)
        regions.append((s0, s1))
    return regions


def analyze_missed_element_region(species_name, signal, sample_rate, s0, s1):
    """Render one confirmed rescue region from the raw signal and run the
    unmodified detection pipeline on it alone. Returns (time_buckets,
    region_start_seconds) in the recording's timeline, or None."""
    crop = signal[s0:s1]
    if len(crop) < 256:
        return None

    tmp_png_path = KATYDIDS_DIR / '_missed_element_rescue_tmp.png'
    try:
        render_raw_crop_png(crop, sample_rate, tmp_png_path)
        spec_array = np.array(Image.open(str(tmp_png_path)).convert('L'))
        duration_seconds = len(crop) / sample_rate
        pixel_time = duration_seconds / spec_array.shape[1]
        center_exclude_fraction = CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name)
        ink = extract_outer_band_ink(spec_array, center_exclude_fraction=center_exclude_fraction)
        local_signal_list, _ = detect_signal_list_adaptive(ink, pixel_time)
    except (ValueError, ZeroDivisionError):
        return None
    finally:
        if tmp_png_path.exists():
            tmp_png_path.unlink()

    signal_columns = np.where(local_signal_list == 1)[0]
    if not len(signal_columns):
        return None
    local_trim_start = max(0, signal_columns[0] - 2)
    local_trim_end   = min(len(local_signal_list), signal_columns[-1] + 3)
    cleaned = clean_signal_runs(
        local_signal_list[local_trim_start:local_trim_end], pixel_time,
        fill_gap_size=FILL_GAP_SIZE_OVERRIDES.get(species_name, 0),
    )
    if not len(cleaned) or cleaned.mean() == 0:
        return None

    leading_offset_s = local_trim_start * pixel_time
    time_buckets = [
        ('On' if value == 1 else 'Off', run_length * pixel_time)
        for value, run_length in rle(cleaned)
    ]
    region_start_s = s0 / sample_rate + leading_offset_s
    return time_buckets, region_start_s


def rescued_regions_to_bursts(regions_with_buckets):
    """Combine every confirmed rescue region's elements into one absolute-time
    sequence and run segment_bursts once across all of them--not per region
    (which would split one coherent phrase into spurious 1-element bursts) and
    never mixed with the primary region's gaps.
    regions_with_buckets: (time_buckets, region_start_s) per region."""
    on_spans = []
    for time_buckets, region_start_s in regions_with_buckets:
        elapsed = 0.0
        for state, dur in time_buckets:
            if state == 'On':
                on_spans.append((region_start_s + elapsed, region_start_s + elapsed + dur))
            elapsed += dur
    if not on_spans:
        return []
    on_spans.sort()

    combined_time_buckets = []
    prev_end = None
    for start, end in on_spans:
        if prev_end is not None:
            combined_time_buckets.append(('Off', start - prev_end))
        combined_time_buckets.append(('On', end - start))
        prev_end = end

    on_durations = [d for s, d in combined_time_buckets if s == 'On']
    element_length = float(np.mean(on_durations))
    _, _, _, bursts = segment_bursts(
        combined_time_buckets, 0.0, 0.0, element_length, 0.002, force_trill=False
    )
    base_start = on_spans[0][0]
    for b in bursts:
        b['start_time'] += base_start
        b['end_time']   += base_start
    return bursts


## Processing Function

`katydid_process_multi` works like the single-burst pipeline's
`katydid_process`, with the low-volume rescue pass added in. The difference:
it always returns a list of results, one per burst type, so callers can
just `rows.extend(...)` regardless of how many types a recording has.

Species-specific overrides carry over unchanged--detection-quality fixes
don't care about burst-type clustering.

In [ ]:
def katydid_process_multi(audio_path):
    """
    Multi-burst-type variant of katydid_process (see
    Processing_Katydid_Spectrograms.ipynb). Identical through binary on/off
    detection -- image generation, ink extraction, adaptive thresholding
    (now with a low-volume rescue pass), silence trimming, RLE into time
    buckets. Diverges after that: segment_bursts() finds each individual
    burst and classify_burst_types() clusters those bursts into distinct
    "types" (e.g. an alternating ABCBA call), instead of classify_intervals'
    single aggregate Elements_Per_Burst for the whole recording.

    Species-specific overrides consulted here: CENTER_EXCLUDE_FRACTION_OVERRIDES,
    FILL_GAP_SIZE_OVERRIDES, LADDER_FILL_GAP_OVERRIDES, BAND_WIDTH_HZ_OVERRIDES,
    RELAXED_BURST_GAP_RATIO_SPECIES, and CONTINUOUS_TRILL_SPECIES -- see

    Returns a LIST of dicts, one per detected burst type in this recording --
    a species with one uniform burst type still returns a single-item list.
    """
    audio_path   = Path(audio_path)
    species_name = audio_path.parent.name

    # ── Resolve manually-cropped substitute, if any ─────────────────────────────
    source_path = Path(KATYDID_CROP_LOOKUP.get(str(audio_path), audio_path))

    # ── Load and crop audio ─────────────────────────────────────────────────────
    raw_signal, sample_rate = librosa.load(str(source_path), sr=None)
    crop_start, crop_end    = compute_signal_crop(raw_signal, sample_rate)
    signal                  = raw_signal[crop_start:crop_end]
    duration_seconds        = len(signal) / sample_rate

    # ── Generate spectrogram image (skip if already saved) ─────────────────────
    spec_path = generated_spectrogram_path(audio_path)
    if not spec_path.exists():
        generate_spectrogram_image(
            signal, sample_rate, spec_path,
            band_width_hz=BAND_WIDTH_HZ_OVERRIDES.get(species_name, 500),
        )

    # ── pixel_time: seconds per pixel width ─────────────────────────────────────
    spec_array = np.array(Image.open(str(spec_path)).convert('L'))
    if spec_array.shape[1] == 0:
        raise ValueError('Empty image')
    pixel_time = duration_seconds / spec_array.shape[1]

    # ── Compute ink from outer bands only (excludes centerline) ─────────────────
    center_exclude_fraction = CENTER_EXCLUDE_FRACTION_OVERRIDES.get(species_name)
    ink = extract_outer_band_ink(spec_array, center_exclude_fraction=center_exclude_fraction)

    # ── Detect binary on/off signal ─────────────────────────────────────────────
    raw_signal_list, _ = detect_signal_list_adaptive(
        ink, pixel_time, fill_gap_size_override=LADDER_FILL_GAP_OVERRIDES.get(species_name)
    )

    # ── Rescue bursts too quiet to survive the global threshold (see markdown) ──
    raw_signal_list = rescue_quiet_bursts(
        spec_array, ink, raw_signal_list, pixel_time,
        center_exclude_fraction=center_exclude_fraction,
    )

    # ── Trim leading and trailing silence ───────────────────────────────────────
    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError('No signal columns detected')

    trim_start = max(0, signal_columns[0]  - 2)
    trim_end   = min(len(raw_signal_list), signal_columns[-1] + 3)

    leading_silence  = trim_start * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time,
        fill_gap_size=FILL_GAP_SIZE_OVERRIDES.get(species_name, 0),
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError('Empty signal after cleanup')

    # ── Build time bucket list: [(state, duration_seconds), ...] ────────────────
    time_buckets = [
        ('On' if value == 1 else 'Off', run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]

    # ── Element length = mean on-run duration ────────────────────────────────────
    on_durations = [duration for state, duration in time_buckets if state == 'On']
    if not on_durations:
        raise ValueError('No on-pulses detected')
    element_length = float(np.mean(on_durations))

    # ── Segment into individual bursts, then cluster bursts into types ──────────
    force_trill = species_name in CONTINUOUS_TRILL_SPECIES
    _, inter_burst_interval, _, bursts = segment_bursts(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time,
        force_trill=force_trill,
        min_gap_ratio=RELAXED_BURST_GAP_RATIO_SPECIES.get(species_name, 5.0),
        force_element_level_split=species_name in ELEMENT_LEVEL_BURST_SPECIES,
    )

    # ── Missed-element rescue (species-specific, see markdown above) ────────────
    if (species_name, audio_path.name) in MISSED_ELEMENT_RESCUE_FILES:
        primary_range = (
            int(trim_start * pixel_time * sample_rate),
            int(trim_end * pixel_time * sample_rate),
        )
        rescue_regions = find_missed_element_regions(signal, sample_rate, primary_range)
        regions_with_buckets = []
        for s0, s1 in rescue_regions:
            result = analyze_missed_element_region(species_name, signal, sample_rate, s0, s1)
            if result is not None:
                regions_with_buckets.append(result)

        # force_trill species must collapse to one burst, so re-segment the full
        # primary+rescued timeline once. Doing this for non-force_trill files
        # changed ones nobody flagged, so they keep the plain concatenation below.
        if regions_with_buckets and force_trill:
            primary_on_spans = []
            elapsed = 0.0
            for state, dur in time_buckets:
                if state == 'On':
                    primary_on_spans.append((leading_silence + elapsed, leading_silence + elapsed + dur))
                elapsed += dur

            rescued_on_spans = []
            for region_buckets, region_start_s in regions_with_buckets:
                r_elapsed = 0.0
                for state, dur in region_buckets:
                    if state == 'On':
                        rescued_on_spans.append((region_start_s + r_elapsed, region_start_s + r_elapsed + dur))
                    r_elapsed += dur

            all_spans = sorted(primary_on_spans + rescued_on_spans)
            combined_time_buckets = []
            prev_end = None
            for start, end in all_spans:
                if prev_end is not None:
                    combined_time_buckets.append(('Off', start - prev_end))
                combined_time_buckets.append(('On', end - start))
                prev_end = end

            combined_element_length = float(np.mean(
                [d for s, d in combined_time_buckets if s == 'On']
            ))
            _, inter_burst_interval, _, bursts = segment_bursts(
                combined_time_buckets, 0.0, 0.0, combined_element_length, pixel_time,
                force_trill=force_trill,
                min_gap_ratio=RELAXED_BURST_GAP_RATIO_SPECIES.get(species_name, 5.0),
                force_element_level_split=species_name in ELEMENT_LEVEL_BURST_SPECIES,
            )
            base_start = all_spans[0][0]
            for b in bursts:
                b['start_time'] += base_start
                b['end_time'] += base_start
        else:
            rescued_bursts = rescued_regions_to_bursts(regions_with_buckets)
            if rescued_bursts:
                bursts = bursts + rescued_bursts
                bursts.sort(key=lambda b: b['start_time'])

    # ── Per-burst dominant frequency, an extra clustering feature ───────────────
    burst_frequencies = compute_burst_frequencies(bursts, signal, sample_rate, time_offset=leading_silence)
    bursts = classify_burst_types(
        bursts,
        max_k=MAX_BURST_TYPES_OVERRIDES.get(species_name, 4),
        force_uniform=(
            species_name in UNIFORM_BURST_TYPE_SPECIES
            # ELEMENT_LEVEL_BURST_SPECIES also forces uniform -- without real
            # burst-level grouping, element-length jitter alone would otherwise
            # get read as distinct "types".
            or species_name in ELEMENT_LEVEL_BURST_SPECIES
        ),
        min_singleton_silhouette=RELAXED_SINGLETON_TYPE_SPECIES.get(species_name, 0.85),
        burst_frequencies=burst_frequencies,
    )

    n_types = len(set(b['burst_type'] for b in bursts))
    pattern = bursts[0]['burst_pattern']

    rows = []
    for burst_type in sorted(set(b['burst_type'] for b in bursts)):
        type_bursts = [b for b in bursts if b['burst_type'] == burst_type]
        counts      = [b['element_count'] for b in type_bursts]
        rows.append({
            'element_length': round(
                float(np.mean([b['mean_element_length'] for b in type_bursts])), 4
            ),
            'inter_element_interval': round(
                float(np.mean([b['mean_inter_element_interval'] for b in type_bursts])), 4
            ),
            'inter_burst_interval':   round(inter_burst_interval, 4),
            'elements_per_burst':     int(statistics.median(counts)),
            'min_elements_per_burst': min(counts),
            'max_elements_per_burst': max(counts),
            'burst_type':             chr(ord('A') + burst_type),
            'n_bursts_this_type':     len(type_bursts),
            'n_burst_types_detected': n_types,
            'burst_pattern':          pattern,
        })
    return rows


## Run Pipeline on All Audio Files

Runs every downloaded katydid file through the pipeline and writes to
`katydid_results_multi_bursts.csv`--a separate file, so this never touches
the single-burst pipeline's output. One file can now produce several rows.
Cropping and image generation reuse the original pipeline's cache
unchanged.

In [ ]:
audio_files = sorted(
    p for p in KATYDIDS_DIR.rglob('*_audio_*')
    if 'checkpoint' not in str(p)
    and p.suffix.lower() in {'.mp3', '.wav', '.ogg'}
)

rows = []
for audio_path in audio_files:
    species_folder = audio_path.parent.name
    if species_folder in EXCLUDED_NON_CALLING_SONG_SPECIES:
        continue
    spec_path = generated_spectrogram_path(audio_path)
    try:
        results = katydid_process_multi(audio_path)
        status  = 'ok'
        error   = ''
    except Exception as e:
        results = [{}]
        status  = 'error'
        error   = str(e)
    for result in results:
        rows.append({
            # 'file' holds the generated spectrogram's name (not the audio filename) so
            # the webscraping merge step below can keep using its existing '_spectrogram_' regex.
            'species': species_folder,
            'file':    spec_path.name,
            'status':  status,
            'error':   error,
            **result,
        })

output_csv  = KATYDIDS_DIR / 'katydid_results_multi_bursts.csv'
import csv
csv_columns = [
    'species', 'file', 'status', 'error',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
    'burst_type', 'n_bursts_this_type', 'n_burst_types_detected', 'burst_pattern',
]
with open(output_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(rows)

successful_rows = [r for r in rows if r['status'] == 'ok']
error_rows      = [r for r in rows if r['status'] == 'error']
n_files         = len({(r['species'], r['file']) for r in rows})
n_multi_type    = len({
    (r['species'], r['file']) for r in rows
    if r['status'] == 'ok' and r.get('n_burst_types_detected', 1) > 1
})
print(f'Processed {n_files} audio files -> {len(rows)} rows '
      f'({len(successful_rows)} ok, {len(error_rows)} errors)')
print(f'{n_multi_type} file(s) had more than one detected burst type')
for r in error_rows:
    print(f"  ERROR {r['species']} / {r['file']}: {r['error']}")

## Species-Specific Override: Eremopedes_covilleae

Same fix as the single-burst notebook (disable gap-filling for this species),
but handled via `LADDER_FILL_GAP_OVERRIDES` instead of a dedicated function--that fits a multi-row-per-file output better.

## Results

In [ ]:
df    = pd.read_csv(output_csv)
df_ok = df[df['status'] == 'ok'].copy()
print(f"{len(df_ok)} rows (across {df_ok[['species','file']].drop_duplicates().shape[0]} files) processed successfully")
df_ok[[
    'species', 'burst_type', 'burst_pattern', 'n_bursts_this_type', 'n_burst_types_detected',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]]

## Merge with Webscraping Data

Same join as the single-burst notebook, on the oscillogram file ID--pandas
handles the one-to-many merge fine now that a file can have multiple rows.

**Run Webscraping.ipynb at least once before this cell.**

Stored as `katydid_final_multi`, kept separate from `katydid_final` so this
notebook never touches the single-burst pipeline's results.

In [ ]:
import os
import re

# Load the webscraping DataFrame from IPython's persistent store.
# Requires that %store katydid_df was run in Webscraping.ipynb beforehand.
%store -r katydid_df


def spec_id_from_url(url):
    """'https://orthsoc.org/sina/010so.jpg' -> '010so'"""
    if pd.isna(url):
        return None
    return os.path.splitext(os.path.basename(str(url)))[0]


def spec_id_from_filename(filename):
    """'Amblycorypha_oblongifolia_spectrogram_010so.jpg' -> '010so'"""
    match = re.search(r'_spectrogram_(.+)\.[^.]+$', str(filename))
    return match.group(1) if match else None


def audio_id_from_url(url):
    """Audio-URL basename, e.g. '.../801-acrolophitus.mp3' -> '801-acrolophitus'.
    Preferred over spec_id for File_ID: many recordings share a generic
    'sound.gif' spectrogram, but Audio_Link basenames are distinct."""
    if pd.isna(url):
        return None
    return os.path.splitext(os.path.basename(str(url)))[0]


# Build join keys
# Note: katydid_df uses 'Description' (not 'Description of Whole Audio File')
webscraping_data = katydid_df[[
    'Species', 'Temperature (\u00b0C)', 'Location',
    'Description', 'Map', 'Spectrogram', 'Audio_Link',
]].copy()
webscraping_data['_spec_id'] = webscraping_data['Spectrogram'].apply(spec_id_from_url)
webscraping_data['_audio_id'] = webscraping_data['Audio_Link'].apply(audio_id_from_url)

# _spec_id isn't unique across species (many share a generic id like 'sound'),
# so the join also keys on _species_key--both sides normalized to
# lowercase-underscore--to keep 'sound' matching only within one species.
webscraping_data['_species_key'] = (
    webscraping_data['Species'].str.strip().str.replace(r'\s+', '_', regex=True).str.lower()
)

processing_results = df_ok[[
    'species', 'file', 'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
    'burst_type', 'n_bursts_this_type', 'n_burst_types_detected', 'burst_pattern',
]].copy()
processing_results['_spec_id'] = processing_results['file'].apply(spec_id_from_filename)
processing_results['_species_key'] = processing_results['species'].str.strip().str.lower()

# Join on oscillogram file ID *and* species, so a generic shared id like 'sound'
# only matches within the same species instead of across all species that used it.
merged = processing_results.merge(
    webscraping_data.drop(columns='Spectrogram'),
    on=['_spec_id', '_species_key'],
    how='left',
)

# Drop processed files with no matching metadata row (e.g. a duplicate
# '..._audio_sound.mp3' with no SINA spectrogram entry)--they'd show as a
# broken "nan / nan" species in the viewer.
unmatched = merged[merged['Species'].isna()]
if len(unmatched):
    print(f'Dropping {len(unmatched)} processed file(s) with no matching webscraping '
          f'metadata (species, spec_id): '
          + ', '.join(f"{r['species']}/{r['_spec_id']}" for _, r in unmatched.iterrows()))
merged = merged[merged['Species'].notna()].copy()

# Split 'Genus species' into two separate columns
merged[['Genus', 'Species']] = merged['Species'].str.split(' ', n=1, expand=True)

# Prefer the audio file's own id (traceable back to the actual source recording);
# fall back to the spectrogram-derived id for the rare rows with no Audio_Link.
merged['File_ID'] = merged['_audio_id'].fillna(merged['_spec_id'])

# Column names must match the viewer exactly. File_ID is the source-recording
# id (from Audio_Link, else spectrogram-derived); Spec_ID is kept separately
# because it--not File_ID--is embedded in the generated PNG filename the
# viewer matches on.
katydid_final_multi = merged[[
    'Genus', 'Species',
    'Temperature (\u00b0C)', 'Location',
    'Description', 'Map',
    'File_ID', '_spec_id',
    'burst_type', 'burst_pattern', 'n_bursts_this_type', 'n_burst_types_detected',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].rename(columns={
    'Temperature (\u00b0C)':        'Temperature',
    '_spec_id':                 'Spec_ID',
    'burst_type':               'Burst_Type',
    'burst_pattern':             'Burst_Pattern',
    'n_bursts_this_type':        'N_Bursts_This_Type',
    'n_burst_types_detected':    'N_Burst_Types_Detected',
    'element_length':           'Element_Length',
    'inter_element_interval':   'Inter-Element_Interval',
    'inter_burst_interval':     'Inter-Burst_Interval',
    'elements_per_burst':       'Elements_Per_Burst',
    'min_elements_per_burst':   'Min_Elements_Per_Burst',
    'max_elements_per_burst':   'Max_Elements_Per_Burst',
})

print(
    f'{len(katydid_final_multi)} rows  |  '
    f'{katydid_final_multi["Genus"].nunique()} genera  |  '
    f'{katydid_final_multi["Species"].nunique()} species  |  '
    f'{(katydid_final_multi["N_Burst_Types_Detected"] > 1).sum()} rows from a '
    f'multi-burst-type recording'
)
katydid_final_multi.head()


In [ ]:
# Store separately from the original pipeline's katydid_final -- this is an
# exploratory alternate output, not consumed by Display_Function.ipynb.
%store katydid_final_multi

In [ ]:
katydid_final_multi.to_csv(Path.home() / 'Discrete_Signals' / 'katydid_multi_burst_type.csv', index=False)